In [27]:
from google.colab import drive
drive.mount("/content/drive")

proj_path = '/content/drive/MyDrive/CSE 3521/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
# right click to upload transformer.py and dataset.py to colab runtime

import sys
sys.path.append('/content')

from fine_tune_dataset import ReviewDataset
from utils import report_metrices

In [29]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

In [30]:
model_name = 'bert-base-uncased'
num_labels = 3

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
for p in model.parameters(): p.requires_grad = False
for layer in model.bert.encoder.layer[-4:]:
    for p in layer.parameters(): p.requires_grad = True
for p in model.bert.pooler.parameters():
    p.requires_grad = True
for p in model.classifier.parameters():
    p.requires_grad = True

In [32]:
trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
enc  = [p for n, p in trainable if n.startswith("bert.")]
head = [p for n, p in trainable if n.startswith("classifier.")]

optimizer = torch.optim.AdamW([
    {"params": [p for p in enc if p.ndim > 1],  "lr": 2e-5, "weight_decay": 0.01},
    {"params": [p for p in enc if p.ndim <= 1], "lr": 2e-5, "weight_decay": 0.0},
    {"params": head,                            "lr": 1e-3, "weight_decay": 0.0},
])

In [ ]:
training_args = TrainingArguments(
    learning_rate=2e-5,                
    num_train_epochs=3,
    per_device_train_batch_size=32,    
    per_device_eval_batch_size=64,

    weight_decay=0.01,                 
    warmup_steps=500,

    dataloader_num_workers=2,

    eval_strategy="steps",
    eval_steps=250,
    logging_steps=50,
    logging_first_step=True,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

)

In [34]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [35]:
train_dataset = ReviewDataset(proj_path + "data/train.csv", tokenizer)
val_dataset = ReviewDataset(proj_path + "data/val.csv", tokenizer)

In [37]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    compute_metrics=compute_metrics,
    optimizers = (optimizer, None)
)


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,Macro F1
250,0.619289,0.588241,0.748057,0.637655
500,0.520133,0.522749,0.784542,0.664733
750,0.477746,0.503496,0.795553,0.703604
1000,0.499776,0.467428,0.805052,0.727864
1250,0.406665,0.468280,0.804620,0.742263
1500,0.444760,0.441020,0.818005,0.753559
1750,0.443932,0.435888,0.819301,0.753463
2000,0.399102,0.458669,0.819085,0.767951
2250,0.406212,0.437091,0.826641,0.765412
2500,0.396846,0.459854,0.819516,0.748259


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (9 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
trainer.save_model(proj_path + "modern/fine_tuned")
tokenizer.save_pretrained(proj_path+"modern/fine_tuned")

AttributeError: 'tokenizers.trainers.BpeTrainer' object has no attribute 'save_model'

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_dir = proj_path + "modern/fine_tuned"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

In [ ]:
import torch, torch.nn.functional as F

@torch.inference_mode()
def predict_batch(texts, model, tokenizer, device, batch_size=64, max_length=256):
    model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(
            texts[i:i + batch_size],
            truncation=True, max_length=max_length,
            padding=True, return_tensors="pt",
        ).to(device)
        logits = model(**batch).logits
        all_probs.append(logits.softmax(dim=-1).cpu())
    return torch.cat(all_probs, dim=0)   

In [ ]:
df = pd.read_csv(proj_path + "data/test.csv")
texts = df["input_text"].astype(str).tolist()
probs = predict_batch(texts, model, tokenizer, device)
y_pred = probs.argmax(dim=-1).tolist()

report_metrices(df['Sentiment'], y_pred)
